In [1]:
!hdfs dfs -D dfs.replication=1 -cp -f data/*.jsonl hdfs://nn:9000/
!hdfs dfs -D dfs.replication=1 -cp -f data/*.csv hdfs://nn:9000/


In [2]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder.appName("cs544")
        .master("spark://boss:7077")
        .config("spark.executor.memory", "1G")
        .config("spark.sql.warehouse.dir", "hdfs://nn:9000/user/hive/warehouse")
        .enableHiveSupport()
        .getOrCreate())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/30 20:44:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#loading the data
problems_df = spark.read.json("hdfs://nn:9000/problems.jsonl")
problems_df.limit(5).show()


[Stage 1:>                                                          (0 + 1) / 1]

+-------------+--------+---------+---------+---------------+----------+---------------+-------------------------+------------------+--------------------+-------------+----------+------------+------+----------+
|cf_contest_id|cf_index|cf_points|cf_rating|        cf_tags|difficulty|generated_tests|is_description_translated|memory_limit_bytes|                name|private_tests|problem_id|public_tests|source|time_limit|
+-------------+--------+---------+---------+---------------+----------+---------------+-------------------------+------------------+--------------------+-------------+----------+------------+------+----------+
|          322|       A|    500.0|     1000|            [0]|         7|             93|                    false|         256000000|322_A. Ciel and D...|           45|         1|           2|     2|         1|
|          760|       D|   1000.0|     1600|         [1, 2]|        10|             51|                    false|         256000000|  760_D. Travel Card|       

In [ ]:
# PART1- Filtering: RDDs, DataFrames, and Spark

In [57]:
#q1: How many problems are there with a cf_rating of at least 1600, having private_tests, and a name containing "_A." (Case Sensitive)? Answer by directly using the RDD API.

output1 = problems_df.rdd.filter(lambda row: 
                                 row['cf_rating'] >= 1600 and 
                                 row['private_tests'] > 0 and 
                                 "_A." in row['name'])

output1.count()

217

In [64]:
#q2:  How many problems are there with a cf_rating of at least 1600, having private_tests, and a name containing "_A." (Case Sensitive)? Answer by using the DataFrame API.
from pyspark.sql.functions import col, expr

output2 = problems_df.filter((col("cf_rating") >= 1600) & 
                              (col("private_tests") > 0) & 
                              expr("name like '%_A.%'")
                             )
output2.count()


217

In [67]:
#q3: How many problems are there with a cf_rating of at least 1600, having private_tests, and a name containing "_A." (Case Sensitive)? Answer by using Spark SQL.
problems_df.write.mode("overwrite").saveAsTable("problems_temp")

# Query the new table
output3 = spark.sql("""
    SELECT COUNT(*) as count
    FROM problems_temp
    WHERE cf_rating >= 1600
    AND private_tests > 0
    AND name LIKE '%_A.%'
""").collect()[0][0]

print(f"{output3}")


217


In [ ]:
# PART2- Hive Data Warehouse

In [19]:
#q4: Does the query plan for a GROUP BY on solutions data need to shuffle/exchange rows if the data is pre-bucketed?

solutions_df = spark.read.json("hdfs://nn:9000/solutions.jsonl")

solutions_df.write \
    .bucketBy(4, "language").mode("overwrite") \
    .saveAsTable("solutions")

explainq = spark.sql("""
    SELECT language, COUNT(*)
    FROM solutions
    GROUP BY language
""")

explainq.explain(True)

[Stage 46:===========================================>              (6 + 2) / 8]

== Parsed Logical Plan ==
'Aggregate ['language], ['language, unresolvedalias('COUNT(1), None)]
+- 'UnresolvedRelation [solutions], [], false

== Analyzed Logical Plan ==
language: string, count(1): bigint
Aggregate [language#731], [language#731, count(1) AS count(1)#734L]
+- SubqueryAlias spark_catalog.default.solutions
   +- Relation spark_catalog.default.solutions[is_correct#730,language#731,problem_id#732L,solution#733] parquet

== Optimized Logical Plan ==
Aggregate [language#731], [language#731, count(1) AS count(1)#734L]
+- Project [language#731]
   +- Relation spark_catalog.default.solutions[is_correct#730,language#731,problem_id#732L,solution#733] parquet

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[language#731], functions=[count(1)], output=[language#731, count(1)#734L])
   +- HashAggregate(keys=[language#731], functions=[partial_count(1)], output=[language#731, count#739L])
      +- FileScan parquet spark_catalog.default.solutions[language

In [26]:
#q5: What tables/views are in our warehouse?

languages_df = spark.read.csv("hdfs://nn:9000/languages.csv", header=True, inferSchema=True)
languages_df.createOrReplaceTempView("languages")

problem_tests_df = spark.read.csv("hdfs://nn:9000/problem_tests.csv", header=True, inferSchema=True)
problem_tests_df.createOrReplaceTempView("problem_tests")

sources_df = spark.read.csv("hdfs://nn:9000/sources.csv", header=True, inferSchema=True)
sources_df.createOrReplaceTempView("sources")

tags_df = spark.read.csv("hdfs://nn:9000/tags.csv", header=True, inferSchema=True)
tags_df.createOrReplaceTempView("tags")

tables= spark.catalog.listTables()

dict = {table.name: table.isTemporary for table in tables}
print(dict)


{'problems': False, 'solutions': False, 'languages': True, 'problem_tests': True, 'sources': True, 'tags': True}


In [ ]:
# PART3- Caching and Transforming Data

In [74]:
#q6: How many correct PYTHON3 solutions are from CODEFORCES?

output6 = spark.sql("""SELECT COUNT(*) as count
    FROM solutions sol
    JOIN problems prob ON sol.problem_id = prob.problem_id
    JOIN sources src ON prob.source = src.source
    WHERE sol.is_correct = true
    AND src.source_name = 'CODEFORCES'
    AND sol.language = 'PYTHON3'
""").first()["count"]

print(output6)

10576


In [70]:
#q7: How many problems are of easy/medium/hard difficulty?

df = spark.sql("""
    SELECT 
        CASE 
            WHEN difficulty <= 5 THEN 'Easy'
            WHEN difficulty <= 10 THEN 'Medium'
            ELSE 'Hard'
        END AS difficulty_category,
        COUNT(*) AS count
    FROM problems
    GROUP BY 
        CASE 
            WHEN difficulty <= 5 THEN 'Easy'
            WHEN difficulty <= 10 THEN 'Medium'
            ELSE 'Hard'
        END
""")

difficulty_dict = {row['difficulty_category']: row['count'] for row in df.collect()}
print(difficulty_dict)


{'Easy': 409, 'Medium': 5768, 'Hard': 2396}


In [47]:
#q8: Does caching make it faster to compute averages over a subset of a bigger dataset?

import time

tests = spark.sql("SELECT * FROM problem_tests WHERE is_generated = False")

start1 = time.time()
result1 = tests.agg({"input_chars": "avg", "output_chars": "avg"}).collect()
time1 = time.time() - start1

tests.cache()

start2 = time.time()
result2 = tests.agg({"input_chars": "avg", "output_chars": "avg"}).collect()
time2 = time.time() - start2

start3 = time.time()
result3 = tests.agg({"input_chars": "avg", "output_chars": "avg"}).collect()
time3 = time.time() - start3

tests.unpersist()

output = [time1, time2, time3]
print(output)




[0.2955024242401123, 0.397860050201416, 0.09754300117492676]


In [44]:
# PART4 - Machine Learning with Spark

In [72]:
#q9: How well can a decision tree predict cf_rating based on difficulty, time_limit, and memory_limit_bytes?

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

codeforces = problems_df.join(sources_df, problems_df.source == sources_df.source) \
    .filter(sources_df.source_name == 'CODEFORCES')

train_df = codeforces.filter("cf_rating > 0 AND problem_id % 2 == 0")
test_df = codeforces.filter("cf_rating > 0 AND problem_id % 2 != 0")

assembler = VectorAssembler(inputCols=['difficulty', 'time_limit', 'memory_limit_bytes'], outputCol='features')

decisiontree = DecisionTreeRegressor(featuresCol='features', labelCol='cf_rating', maxDepth=5)

pipeline = Pipeline(stages=[assembler, decisiontree])

# Fitting the model on the training data
model = pipeline.fit(train_df)
predictions = model.transform(test_df)

evaluator = RegressionEvaluator(labelCol='cf_rating', predictionCol='prediction', metricName='r2')
R2 = evaluator.evaluate(predictions)

print(R2)


0.5929835263198762


In [55]:
#q10: Do the problems with a missing cf_score appear more or less challenging that other problems?

avg_train = train_df.selectExpr("avg(cf_rating)").collect()[0][0]
avg_test= test_df.selectExpr("avg(cf_rating)").collect()[0][0]
missing_pred = model.transform(missing_df)
avg_missing = missing_pred.selectExpr("avg(prediction)").collect()[0][0]
output = (avg_train, avg_test, avg_missing)
print(output)


(1887.9377431906614, 1893.1106471816283, 1950.4728638818783)
